# Project Part 3 - Mitra Iyer 10/15/25

## The purpose of this project is to visualize the data and answer the data science questions through various plots

### Data Science Questions
     1. What are the order courses that students enrolled in had the most participants in campus events or activities?
    2. What is the order of majors that havethe most students who were exposed to computing classes in middle or high school?
    3. What are the majors that have the fewest people who identify as a woman?
    4. From which source did the most students receive information about CCM’s computing programs?  Which source did the least students hear information about CCM's computing programs?
    5. Which age group is least represented in computing

#### imports

In [ ]:
import pandas as pd

In [ ]:
import numpy as np

In [ ]:
import matplotlib.pyplot as plt

In [ ]:
data = pd.read_csv('MajorSurveyResults24_cleaned_data.csv') 

### Question 1: What are the top courses that students enrolled in had the most participants in campus events or activities?


#### combine all the event columns into one 

In [ ]:
event_cols = [col for col in data.columns if col.startswith('event_')]

In [ ]:
data.columns

In [ ]:
event_cols

### Normalizes the data: I added this because it kept saying that the data was not being properly summed, which meant that when converting to yes/no, there were external chracters and it kept giving me errors

In [ ]:
for col in event_cols:
    data[col] = data[col].astype(str).str.strip().str.lower()

### Now I loop through the columns, where I made "yes" as a value 1, and no/not sure as a value 0 so I can sum the values

In [ ]:
for col in event_cols:
    data[col] = data[col].replace({"yes": 1, "no": 0, "not sure": 0}).astype(int)

### I made the data for event columns integers so I can add them

In [ ]:
data[event_cols] = data[event_cols].astype(int)

### I added up all the sums as total events

In [ ]:
data["total_events"] = data[event_cols].sum(axis=1)

### Now I group each course for total_events as sum and sort the values from dscending order

In [ ]:
course_event_counts = data.groupby("course_enrolled")["total_events"].sum().sort_values(ascending=False)

In [ ]:
course_event_counts

### Plot the values in the graph using matplotlib

In [ ]:
plt.figure(figsize=(12,6)) #figure size
bars = plt.bar(course_event_counts.index, course_event_counts.values,color='teal', edgecolor='black')  #added color

#title & x & y lables
plt.title("Total Campus Event Participation by Course", fontsize=14)
plt.xlabel("Course Enrolled")
plt.ylabel("Total Event Participation")
plt.xticks(rotation=45, ha='right') #rotate the course names

#add values to the bars
for bar in bars:
    height = bar.get_height()
    plt.text(bar.get_x() + bar.get_width()/2, height + 1, f'{int(height)}',
             ha='center', va='bottom', fontsize=10)

plt.axhline(0, color='gray', linewidth=1)#added baseline
plt.tight_layout() #tight layout so graph is not too big
plt.show() #shows the graph

#### Analysis: CMP 128 Computer Science I students participate in the most eents, while those taking CMP 130 Intro to IT participate in the least events

## Question 2:
    What is the order of majors that have the most students who were exposed to computing classes in middle or high school?


### First I put all the impact columns into one variable

In [ ]:
impact_columns = [col for col in data.columns if col.startswith("impact_")]

In [ ]:
### Then I create a map for the values that converts them to integer values that I can sum

In [ ]:
impact_map = {
    "No Impact": 0,
    "Some Impact": 1,
    "High Impact": 1,
    "No Response": 0,
    "Don't Recall": 0,
    np.nan: 0
}

data[impact_columns] = data[impact_columns].replace(impact_map).astype(int)

### Then I marked students as exposed if there was any impact (so impact was greater than 0 (gt(0) checks if any cell is greater than 0)

In [ ]:
data["exposed"] = data[impact_columns].gt(0).any(axis=1)

### Then I grouped the data by the major and I counted how many students were exposed

In [ ]:
exposure_by_major = (
    data[data["exposed"]]
    .groupby("degree_program")
    .size()
    .sort_values(ascending=False)
)

### Now I print the exposure by major to make sure data makes sense

In [ ]:
exposure_by_major

### Plot the data in a bar graph so I can compare the data *Note, bar graph only shows majors with totals > 1 for clarity

In [ ]:
#filters for majors greater than 1
exposure_filtered = exposure_by_major[exposure_by_major > 1]

#plot size
plt.figure(figsize=(12,6))
bars = plt.bar(exposure_filtered.index, exposure_filtered.values, color='teal', edgecolor='black') #puts in the values and index based on the filter, and colors it teal and black

#labels & title
plt.title("Exposure to Computing Classes by Major in Highschool/Middleschool (Exposures > 1)", fontsize=14)
plt.xlabel("Major")
plt.ylabel("Number of Students Exposed")
plt.xticks(rotation=45, ha='right') #ticks

#values on top of the bars
for bar in bars:
    height = bar.get_height()
    plt.text(bar.get_x() + bar.get_width()/2, height + 0.1, f'{int(height)}', ha='center', va='bottom', fontsize=10)
#layout
plt.axhline(0, color='gray', linewidth=1)
plt.tight_layout()
plt.show()


#### Analysis: More people who are majoring in Computer Science have exposure to technology prior to college compared to other majors/ Cyber Secuirty seems to have the least (when exposure is > 1)

## Question 3: 
     What are the majors that have the fewest people who identify as woman? What about highest concentration of women?

### Filter the data based on gender 

In [ ]:
filtered = data[data["gender"].isin(["Woman"])]

### sum up the gender count based on the degree and sourt the values

In [ ]:
gender_counts = filtered.groupby("degree_program").size().sort_values(ascending=False)

In [ ]:
gender_counts

## Plot the values in a bar graph to compare the majors

In [ ]:
plt.figure(figsize=(12,6)) #dimensions
bars = plt.bar(gender_counts.index, gender_counts.values, color='red', edgecolor='black')

plt.title("Number of Women by Major", fontsize=14)
plt.xlabel("Major")
plt.ylabel("Count of Women/Non-Binary Students")
plt.xticks(rotation=45, ha='right')

#add values on the bars
for bar in bars:
    height = bar.get_height()
    plt.text(bar.get_x() + bar.get_width()/2, height + 0.1, f'{int(height)}', ha='center', va='bottom', fontsize=10)

plt.axhline(0, color='gray', linewidth=1)
plt.tight_layout()
plt.show()

#### Analysis: Computer Science has the highest concentration of women compared to degrees like Cybersecurity, Buisness Adminstration, etc which have only one woman. 

## Question 4:
    From which source did the most students receive information about CCM’s computing programs?  Which source did the least students hear information about CCM's computing programs?

### combine all the info columns

In [ ]:
info_columns = [col for col in data.columns if col.startswith("info_")]

### count how many students said yes in each column

In [ ]:
for col in info_columns:
    data[col] = data[col].astype(str).str.strip().str.lower()

In [ ]:
for col in info_columns:
    data[col] = data[col].replace({"yes": 1, "no": 0, "don't recall": 0, "no response": 0}).astype(int)

In [ ]:
info_counts = data[info_columns].sum().sort_values(ascending=False)

In [ ]:
info_counts

### Bar Plot for comparisons

In [ ]:
plt.figure(figsize=(10,5))
bars = plt.bar(info_counts.index, info_counts.values, color='skyblue', edgecolor='black')

plt.title("Number of Students Who Heard About CCM's Computing Programs by Source", fontsize=14)
plt.xlabel("Source")
plt.ylabel("Number of Students")
plt.xticks(rotation=45, ha='right')

# Add values on top of bars
for bar in bars:
    height = bar.get_height()
    plt.text(bar.get_x() + bar.get_width()/2, height + 0.1, f'{int(height)}', ha='center', va='bottom', fontsize=10)

plt.axhline(0, color='gray', linewidth=1)
plt.tight_layout()
plt.show()

#### Analysis: Most people hear about CCM Computing Programs from the admissions departments and the CCM website, but the least heard about it from NJ Workforce Development